In [1]:
%load_ext autoreload
%autoreload 2

# Quantum vs CPLEX Results Analysis

This notebook reproduces the `prepare_instances` workflow for 10 protein-ligand complexes,
then computes the docking RMSD of the poses found by the quantum algorithm and CPLEX.

**Output**: a CSV with full name, PDB code, optimality gap, quantum RMSD, CPLEX RMSD.

> **Note on RMSD values**: `place_ligand_from_contacts` prints an internal "Best solution: RMSD"
> which is the **alignment residual (RSSD)** of the contact points via the Kabsch algorithm.
> The final **docking RMSD** is a separate all-atom comparison against the crystal pose (no alignment).

In [2]:
import sys
sys.path.append("..")

import ast
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem

from src.mol_processing.mol import get_protein_from_pdb_file, generate_aligned_conformers
from src.mol_processing.features import extract_pharmacophores, find_pharmacophore_distances
from src.graph.interaction_graph import filter_receptor_features_close_to_ligand, build_distance_matrix
from src.mol_processing.surface import select_surface_features
from src.mol_processing.postprocessing import place_ligand_from_contacts

warnings.filterwarnings("ignore")

RESULTS_DIR = Path("../data/results")

In [3]:
# The 10 instances of interest (from the experiment figure)
INSTANCES = [
    {"code": "2oi0", "name": "TACE", "nodes": 540, "edges": None},
    {"code": "1hwi", "name": "Carbonic anhydrase II", "nodes": 540, "edges": 88600},
    {"code": "1gpk", "name": "Dihydrofolate reductase", "nodes": 253, "edges": 23953},
    {"code": "1xm6", "name": "Retinoic acid receptor", "nodes": 225, "edges": 17806},
    {"code": "1y6b", "name": "Factor VIIa", "nodes": 558, "edges": 99059},
    {"code": "2br1", "name": "Aldose reductase", "nodes": 405, "edges": 57987},
    {"code": "1sqn", "name": "Neuraminidase", "nodes": 451, "edges": 83436},
    {"code": "1yqy", "name": "Adenosine deaminase", "nodes": 585, "edges": 114429},
    {"code": "1w2g", "name": "GSK-3 beta", "nodes": 448, "edges": 73194},
    {"code": "1r58", "name": "PDE4B", "nodes": 570, "edges": 104320},
]

In [4]:
def calculate_docking_rmsd(ref_mol, prb_mol):
    """
    Calculate the docking RMSD between the crystal ligand and the placed ligand
    without any alignment. Same function as in prepare_instances notebook.
    """
    if ref_mol.GetNumAtoms() != prb_mol.GetNumAtoms():
        raise ValueError("Molecules must have the same number of atoms.")
    ref_coords = ref_mol.GetConformer(0).GetPositions()
    prb_coords = prb_mol.GetConformer(0).GetPositions()
    return float(np.sqrt(np.mean(np.sum((ref_coords - prb_coords) ** 2, axis=1))))


def parse_contacts(solution_str):
    """Parse a solution string into a list of (ligand_feature, receptor_feature) tuples."""
    contact_list = ast.literal_eval(solution_str)
    return [(c.split("-")[0], c.split("-")[1]) for c in contact_list]


def load_best_results(code, nodes, edges):
    """Load the result CSV and return the best quantum solution, greedy value, and CPLEX solution."""
    qgc_dir = RESULTS_DIR / "quantum_greedy_cplex"

    if code == "2oi0":
        filepath = qgc_dir / "paper_solution_TACE_AS_COMPL_test_exploration_size_540_.csv"
    else:
        filepath = qgc_dir / f"EXPERIMENT_2000_shots_AnalogDevice_solution_NEW_COMPLEXES_nodes_{nodes}_edges_{edges}_.csv"

    df = pd.read_csv(filepath)

    # Pick the row with the highest quantum_value
    best_idx = df["quantum_value"].idxmax()
    best_row = df.loc[best_idx]

    return {
        "quantum_value": best_row["quantum_value"],
        "greedy_value": best_row["greedy_value"],
        "cplex_value": best_row["opt_value"],
        "solution_quantum": best_row["solution_quantum"],
        "solution_cplex": best_row["solution_cplex"],
    }


def load_sa_value(code, nodes, edges):
    """Load the simulated annealing value for a given instance."""
    sa_dir = RESULTS_DIR / "simulated_annealing"
    sa_files = [
        sa_dir / "EXPERIMENT_SA_solution_firs_experiment_.csv",
        sa_dir / "EXPERIMENT_SA_solution_new_complexes_optimized_second_version.csv",
        sa_dir / "EXPERIMENT_SA_solution_new_small_complexes_optimized_.csv",
    ]

    # For 2oi0: complement edges = 540*539/2 - 44876 = 100654
    if code == "2oi0":
        sa_edges = 100654
    else:
        sa_edges = edges

    for filepath in sa_files:
        df = pd.read_csv(filepath)
        match = df[(df["number_nodes"] == nodes) & (df["number_edges"] == sa_edges)]
        if not match.empty:
            return float(match.iloc[0]["sa_value"])

    return None

## Processing the 10 complexes

For each complex:
1. Load receptor and crystal ligand
2. Extract pharmacophoric features (ligand + receptor)
3. Filter receptor features near the docking site
4. Generate aligned conformers for the ligand
5. Place the ligand using quantum and CPLEX contact solutions
6. Compute docking RMSD vs the crystallographic pose

In [5]:
import numpy as np
from rdkit import Chem
from pathlib import Path


# ---------------------------------------------------------------------------
# Fnat utilities
# ---------------------------------------------------------------------------


def get_native_contacts(receptor, ligand, distance_threshold=4.0):
    """
    Extract native contacts between two RDKit Mol objects.
    Returns a set of (receptor_atom_idx, ligand_atom_idx) tuples
    where heavy-atom distance <= threshold.
    """
    conf_rec = receptor.GetConformer()
    conf_lig = ligand.GetConformer()

    receptor_atoms = []
    for i in range(receptor.GetNumAtoms()):
        if receptor.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        pos = conf_rec.GetAtomPosition(i)
        receptor_atoms.append((i, np.array([pos.x, pos.y, pos.z])))

    ligand_atoms = []
    for i in range(ligand.GetNumAtoms()):
        if ligand.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        pos = conf_lig.GetAtomPosition(i)
        ligand_atoms.append((i, np.array([pos.x, pos.y, pos.z])))

    native_contacts = set()
    for r_idx, r_pos in receptor_atoms:
        for l_idx, l_pos in ligand_atoms:
            if np.linalg.norm(r_pos - l_pos) <= distance_threshold:
                native_contacts.add((r_idx, l_idx))

    return native_contacts


def compute_fnat(native_contacts, predicted_ligand, receptor, distance_threshold=4.0):
    """
    Fnat = |predicted_contacts ∩ native_contacts| / |native_contacts|
    """
    if not native_contacts:
        return None
    predicted_contacts = get_native_contacts(receptor, predicted_ligand, distance_threshold)
    recovered = native_contacts & predicted_contacts
    return len(recovered) / len(native_contacts)


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------

results = []

for instance in INSTANCES:
    code = instance["code"]
    name = instance["name"]
    nodes = instance["nodes"]
    edges = instance["edges"]

    print(f"\n{'=' * 60}")
    print(f"Processing {code.upper()} ({name}) - {nodes} nodes")
    print(f"{'=' * 60}")

    # 1. Load receptor and ligand
    if code == "2oi0":
        receptor = get_protein_from_pdb_file("../data/receptors/2oi0.pdb")
        ligand = Chem.SDMolSupplier("../data/ligands/283_model.sdf", sanitize=True, removeHs=False)[
            0
        ]
    else:
        base_path = Path("../benchmark_complexes")
        receptor = get_protein_from_pdb_file(str(base_path / f"{code}_protein.pdb"))
        ref_mols = [
            m
            for m in Chem.SDMolSupplier(
                str(base_path / f"{code}_ligand_crystal.sdf"),
                sanitize=True,
                removeHs=False,
            )
            if m is not None
        ]
        ligand = ref_mols[0]

    # 2. Native contacts from crystal pose
    native_contacts = get_native_contacts(receptor, ligand)
    print(f"  Native contacts (crystal): {len(native_contacts)}")

    # 3. Extract pharmacophoric features
    ligand_features = extract_pharmacophores(ligand, "l")
    receptor_features_all = extract_pharmacophores(receptor, "R")
    print(
        f"  Ligand features: {len(ligand_features)}, "
        f"Receptor features: {len(receptor_features_all)}"
    )

    # 4. Filter receptor features (docking site + surface)
    _, docking_site_features = filter_receptor_features_close_to_ligand(
        ligand_features, R_features=receptor_features_all, max_distance=6.5
    )
    receptor_features = select_surface_features(receptor, docking_site_features, ligand=ligand)
    print(
        f"  Docking site features: {len(docking_site_features)}, "
        f"Surface features: {len(receptor_features)}"
    )

    # 5. Generate aligned conformers
    ligand_conformers = generate_aligned_conformers(ligand)

    # 6. Load quantum and CPLEX results
    result_data = load_best_results(code, nodes, edges)
    quantum_contacts = parse_contacts(result_data["solution_quantum"])
    cplex_contacts = parse_contacts(result_data["solution_cplex"])

    gap = (result_data["cplex_value"] - result_data["quantum_value"]) / result_data["cplex_value"]
    print(
        f"  Quantum weight: {result_data['quantum_value']:.4f}, "
        f"CPLEX weight: {result_data['cplex_value']:.4f}"
    )
    print(f"  Quantum contacts: {len(quantum_contacts)}, CPLEX contacts: {len(cplex_contacts)}")

    # 7. Place ligand — quantum
    print("  Placing ligand (quantum)...")
    quantum_ligand = place_ligand_from_contacts(
        ligand_conformers, receptor_features_all, quantum_contacts
    )
    rmsd_quantum = calculate_docking_rmsd(ligand, quantum_ligand) if quantum_ligand else None
    fnat_quantum = (
        compute_fnat(native_contacts, quantum_ligand, receptor) if quantum_ligand else None
    )

    # 8. Place ligand — CPLEX
    print("  Placing ligand (CPLEX)...")
    cplex_ligand = place_ligand_from_contacts(
        ligand_conformers, receptor_features_all, cplex_contacts
    )
    rmsd_cplex = calculate_docking_rmsd(ligand, cplex_ligand) if cplex_ligand else None
    fnat_cplex = compute_fnat(native_contacts, cplex_ligand, receptor) if cplex_ligand else None

    # 9. Print results
    print(f"  => Optimality gap: {gap:.4f}")
    if rmsd_quantum is not None:
        print(f"  => Quantum  RMSD: {rmsd_quantum:.3f} A  |  Fnat: {fnat_quantum:.3f}")
    else:
        print("  => Quantum: FAILED")
    if rmsd_cplex is not None:
        print(f"  => CPLEX    RMSD: {rmsd_cplex:.3f} A  |  Fnat: {fnat_cplex:.3f}")
    else:
        print("  => CPLEX: FAILED")

    results.append(
        {
            "Full name": name,
            "PDB code": code.upper(),
            "Optimality gap": round(gap, 4),
            "RMSD Quantum (A)": (round(rmsd_quantum, 3) if rmsd_quantum is not None else None),
            "RMSD CPLEX (A)": (round(rmsd_cplex, 3) if rmsd_cplex is not None else None),
            "Fnat Quantum": (round(fnat_quantum, 3) if fnat_quantum is not None else None),
            "Fnat CPLEX": (round(fnat_cplex, 3) if fnat_cplex is not None else None),
        }
    )



Processing 2OI0 (TACE) - 540 nodes
  Native contacts (crystal): 58
  Ligand features: 15, Receptor features: 1360
6.5
Probe radius calculated from ligand: 2.00 Å
  Docking site features: 41, Surface features: 36
Generated and aligned 100 conformers on the specified core.
  Quantum weight: 5.4040, CPLEX weight: 5.4040
  Quantum contacts: 18, CPLEX contacts: 18
  Placing ligand (quantum)...
Best solution: RMSD = 5.560 Å for conformer 39
  Placing ligand (CPLEX)...
Best solution: RMSD = 6.199 Å for conformer 23
  => Optimality gap: -0.0000
  => Quantum  RMSD: 7.643 A  |  Fnat: 0.121
  => CPLEX    RMSD: 7.670 A  |  Fnat: 0.103

Processing 1HWI (Carbonic anhydrase II) - 540 nodes
  Native contacts (crystal): 76
  Ligand features: 18, Receptor features: 3007
6.5
Probe radius calculated from ligand: 2.07 Å
  Docking site features: 47, Surface features: 30


[19:02:22] Molecule does not have explicit Hs. Consider calling AddHs()
[19:02:23] Molecule does not have explicit Hs. Consider calling AddHs()


  Quantum weight: 12.2107, CPLEX weight: 12.8183
  Quantum contacts: 22, CPLEX contacts: 24
  Placing ligand (quantum)...
Best solution: RMSD = 7.669 Å for conformer 4
  Placing ligand (CPLEX)...
Best solution: RMSD = 11.953 Å for conformer 52
  => Optimality gap: 0.0474
  => Quantum  RMSD: 10.078 A  |  Fnat: 0.053
  => CPLEX    RMSD: 10.283 A  |  Fnat: 0.171

Processing 1GPK (Dihydrofolate reductase) - 253 nodes


FileNotFoundError: [Errno 2] No such file or directory: '../benchmark_complexes/1gpk_protein.pdb'

In [ ]:
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

output_path = "../data/results/comparison_quantum_cplex.csv"
df_results.to_csv(output_path, index=False)
print(f"\nCSV saved to {output_path}")

## Comparison across all methods: Quantum, Greedy, Simulated Annealing

Optimality gap = `(cplex - method) / cplex` for each method (0 = optimal, 1 = worst).

Relative difference of quantum vs another method = `(quantum_gap - other_gap) / other_gap`
(negative means quantum is closer to optimal).

In [ ]:
comparison_rows = []

for instance in INSTANCES:
    code = instance["code"]
    name = instance["name"]
    nodes = instance["nodes"]
    edges = instance["edges"]

    # Load quantum/greedy/cplex values
    result_data = load_best_results(code, nodes, edges)
    cplex_val = result_data["cplex_value"]
    quantum_val = result_data["quantum_value"]
    greedy_val = result_data["greedy_value"]

    # Load simulated annealing value
    sa_val = load_sa_value(code, nodes, edges)

    # Optimality gaps: (cplex - method) / cplex
    gap_quantum = (cplex_val - quantum_val) / cplex_val
    gap_greedy = (cplex_val - greedy_val) / cplex_val
    gap_sa = (cplex_val - sa_val) / cplex_val if sa_val is not None else None

    comparison_rows.append({
        "PDB code": code.upper(),
        "Full name": name,
        "CPLEX weight": round(cplex_val, 4),
        "Quantum weight": round(quantum_val, 4),
        "Greedy weight": round(greedy_val, 4),
        "SA weight": round(sa_val, 4) if sa_val else None,
        "Gap Quantum": round(gap_quantum, 4),
        "Gap Greedy": round(gap_greedy, 4),
        "Gap SA": round(gap_sa, 4) if gap_sa is not None else None,
    })

df_comp = pd.DataFrame(comparison_rows)

# --- Average gaps ---
avg_gap_quantum = df_comp["Gap Quantum"].mean()
avg_gap_greedy = df_comp["Gap Greedy"].mean()
avg_gap_sa = df_comp["Gap SA"].mean()

print("=" * 60)
print("Average optimality gaps (lower is better)")
print("=" * 60)
print(f"  Quantum:             {avg_gap_quantum:.4f}")
print(f"  Greedy:              {avg_gap_greedy:.4f}")
print(f"  Simulated Annealing: {avg_gap_sa:.4f}")

# --- Relative difference: quantum vs greedy, quantum vs SA ---
# (quantum_gap - other_gap) / other_gap  (negative = quantum closer to optimal)
rel_diff_vs_greedy = (avg_gap_quantum - avg_gap_greedy) / avg_gap_greedy
rel_diff_vs_sa = (avg_gap_quantum - avg_gap_sa) / avg_gap_sa

print(f"\nRelative difference of Quantum vs Greedy:  {rel_diff_vs_greedy:+.4f}  ({rel_diff_vs_greedy:+.1%})")
print(f"Relative difference of Quantum vs SA:      {rel_diff_vs_sa:+.4f}  ({rel_diff_vs_sa:+.1%})")

print("\n" + "=" * 60)
print("Per-instance detail")
print("=" * 60)
print(df_comp[["PDB code", "Full name", "Gap Quantum", "Gap Greedy", "Gap SA"]].to_string(index=False))